In [1]:
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv
import sys 
from pathlib import Path
import os

# ------------------------------ #
# Must run together to access NASS API key
# ------------------------------ #
src_path = Path.cwd() / 'src'
sys.path.insert(0, str(src_path))
from utils.nass_api import NASS_API_KEY
# ------------------------------ #

## Prepare Annual Cattle Sales Data

Pulls cattle sales data from the USDA NASS API, filters to exclude calves, cleans the `Value` column, and saves the result to CSV.

In [2]:
base_url = "https://quickstats.nass.usda.gov/api/api_GET/"

params = {'key': NASS_API_KEY,
          'source_desc': 'SURVEY',
          'sector_desc': 'ANIMALS & PRODUCTS',
          'group_desc': 'LIVESTOCK',
          'commodity_desc': 'CATTLE',
          'statisticcat_desc': 'SALES',
          'agg_level_desc': 'NATIONAL'
         }

response = requests.get(base_url, params=params)

cattle_head = response.json()

cattle_head = pd.DataFrame(cattle_head['data'])

## Filter and Clean Cattle Sales Data

In [3]:
# Keep only cattle sold by head count, excluding calves
cattle_head = cattle_head[(cattle_head['unit_desc'] == 'HEAD') & (cattle_head['class_desc'] == '(EXCL CALVES)')]
cattle_head.reset_index(inplace=True, drop=True)

In [4]:
f"Distinct short_desc values: {cattle_head['short_desc'].unique()}"

"Distinct short_desc values: ['CATTLE, (EXCL CALVES), (EXCL INTER-FARM IN-STATE) - SALES, MEASURED IN HEAD']"

In [5]:
select_vars = ['year', 'reference_period_desc', 'begin_code', 'freq_desc',  'unit_desc', 'sector_desc',  'class_desc', 'commodity_desc', 'short_desc', 'statisticcat_desc', 'Value']

cattle_head = cattle_head[select_vars]

In [6]:
print(f"Unique years in cattle_head: {cattle_head['year'].unique()}")

Unique years in cattle_head: [2024 2023 2022 2021 2020 2019 2018 2017 2016 2015 2014 2013 2012 2011
 2010 2009 2008 2007 2006 2005 2004 2003 2002 2001 2000 1999 1998 1997
 1996 1995 1994 1993 1992 1991 1990 1989 1988]


In [7]:
value_clean = cattle_head['Value'].fillna('').astype(str)

# Remove any row containing (D) or (Z) anywhere
mask = value_clean.str.contains(r'\([DZ]\)', regex=True)
cattle_head = cattle_head[~mask].copy()

# Remove ',' from `Value` column
cattle_head['Value'] = (
    cattle_head['Value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
# Convert to numeric
cattle_head['Value'] = pd.to_numeric(cattle_head['Value'], errors='coerce')

# Check for NAs
print(cattle_head['Value'].isna().sum())

# Optional: Drop NAs if needed
# cattle_head.dropna(subset=['Value'], inplace=True) 

0


In [8]:
cattle_head['year'] = cattle_head['year'].astype(int)  # ensure integer type for year

In [9]:
cattle_head = cattle_head.rename(columns={'Value': 'num_head'})  # descriptive column name

In [10]:
## Save to CSV

# Save cleaned data to the shared data directory
cattle_head.to_csv('data/cattle_sales_head_annual_nass.csv', index=False)